# Centralized Analysis for fl-tabular

This notebook is a centralized baseline for comparison with the Flower run. It uses the exact train, validation, and test datasets, drops `CRF01` from the analysis, trains one model end to end, prints progress during training, saves the final model, and evaluates the saved model on the held-out test split.

In [66]:
# ----------------------------- Import Libraries ----------------------------- #
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from fltabular.task import CostRegressor, IGNORED_COLUMNS, _build_preprocessor, evaluator, get_input_dim
# -------------------------- Define Hyperparameters -------------------------- #
BATCH_SIZE = 16
NUM_EPOCHS = 150
LEARNING_RATE = 0.01
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [67]:
#--------------------------------- Load Data -------------------------------- #
def find_file_upwards(filename: str) -> Path:
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / filename
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find {filename} in {Path.cwd()} or any parent directory.')
train_path = find_file_upwards('train_db.csv')
val_path = find_file_upwards('val_db.csv')
test_path = find_file_upwards('test_db.csv')
print(f'Using device: {DEVICE}')
print(f'Train data: {train_path}')
print(f'Val data:   {val_path}')
print(f'Test data:  {test_path}')

Using device: cpu
Train data: /workspaces/flower/fl-tabular/train_db.csv
Val data:   /workspaces/flower/fl-tabular/val_db.csv
Test data:  /workspaces/flower/fl-tabular/test_db.csv


In [68]:
# ---------------------------- Data Preprosessing ---------------------------- #
def _prepare_frame(dataset: pd.DataFrame):
    dataset = dataset.dropna().reset_index(drop=True)

    dataset = dataset.astype({
        "CRF05": "category",
        "CRF10": "category",
        "CRF11": "category",
        "CRF12": "category",
        "CRF13": "category",
        "CRF14": "category",
        "CRF17B": "category",
        "CRF17C": "category",
        "CRF18": "category",
        "CRF19": "category",
        "CRF57": "category",
    })

    target_column = "COST_BL"

    feature_frame = dataset.drop(columns=[target_column], errors="ignore")
    feature_frame = feature_frame.drop(columns=list(IGNORED_COLUMNS), errors="ignore")

    y = pd.to_numeric(dataset[target_column], errors="coerce")

    valid_rows = y.notna()
    feature_frame = feature_frame.loc[valid_rows].reset_index(drop=True)
    y = y.loc[valid_rows].reset_index(drop=True)

    if feature_frame.empty:
        raise ValueError("A dataset split has no usable rows after target filtering.")

    return feature_frame, y

In [69]:
# ----------------------- Data Loaders and Preprocessor ---------------------- #
def load_exact_splits(batch_size: int = BATCH_SIZE):
    train_dataset = pd.read_csv(train_path)
    val_dataset = pd.read_csv(val_path)
    test_dataset = pd.read_csv(test_path)

    x_train_frame, y_train = _prepare_frame(train_dataset)
    x_val_frame, y_val = _prepare_frame(val_dataset)
    x_test_frame, y_test = _prepare_frame(test_dataset)

    preprocessor = _build_preprocessor(x_train_frame)
    x_train = preprocessor.fit_transform(x_train_frame)
    x_val = preprocessor.transform(x_val_frame)
    x_test = preprocessor.transform(x_test_frame)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(x_train, dtype=torch.float32),
            torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1),
        ),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(x_val, dtype=torch.float32),
            torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1),
        ),
        batch_size=batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        TensorDataset(
            torch.tensor(x_test, dtype=torch.float32),
            torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1),
        ),
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, val_loader, test_loader, preprocessor

In [70]:
# ------------------------------- Model Define ------------------------------- #
def train_centralized(model, train_loader, val_loader, 
                      num_epochs: int = NUM_EPOCHS, 
                      learning_rate: float = LEARNING_RATE):
    criterion = nn.L1Loss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    history = []
    model.train()
    for epoch in range(1, num_epochs + 1):
        running_loss = 0.0
        sample_count = 0
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            batch_size = y_batch.size(0)
            running_loss += loss.item() * batch_size
            sample_count += batch_size

        train_loss = running_loss / sample_count
        train_mae, train_mse, train_rmse, train_r2 = evaluator(model, train_loader)
        val_mae, val_mse, val_rmse, val_r2 = evaluator(model, val_loader)
        history.append((train_loss, train_mae, train_mse, train_rmse, 
                        train_r2, val_mae, val_mse, val_rmse, val_r2))
        print(
            f'Epoch {epoch:03d}/{num_epochs} | '
            f'train_loss={train_loss:.4f} | '
            f'train_r2={train_r2:.4f} | '
            f'val_mae={val_mae:.4f} | '
            f'val_rmse={val_rmse:.4f} | '
            f'val_r2={val_r2:.4f}'
        )

    return history

In [71]:
# -------------- Size of the dataset splits and input dimension: ------------- #
train_loader, val_loader, test_loader, preprocessor = load_exact_splits(batch_size=BATCH_SIZE)

print(f'Train samples: {len(train_loader.dataset)}')
print(f'Validation samples: {len(val_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Input dimension: {get_input_dim()}')

Train samples: 256
Validation samples: 65
Test samples: 81
Input dimension: 13


In [72]:
# -------------- Feature inspection of the fitted preprocessor: -------------- #
try:
    original_features = list(preprocessor.feature_names_in_)
except AttributeError:
    original_features = []
    for name, trans, cols in preprocessor.transformers_:
        if cols == 'remainder':
            continue
        original_features.extend(list(cols))

print("Original feature columns:", original_features)

# group columns by transformer name
cat_cols, num_cols = [], []
for name, trans, cols in preprocessor.transformers_:
    if cols == 'remainder':
        continue
    cols_list = list(cols)
    if name.lower().startswith('cat') or 'Ordinal' in trans.__class__.__name__:
        cat_cols.extend(cols_list)
    else:
        num_cols.extend(cols_list)

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# # show categories for ordinal encoders (if present)
# for name, trans, cols in preprocessor.transformers_:
#     if hasattr(trans, "categories_"):
#         for col_name, cats in zip(cols, trans.categories_):
#             print(f"Categories for {col_name}: {list(cats)[:50]}")  # limit output

# # transformed feature names (sklearn >=1.0)
# try:
#     tf_names = preprocessor.get_feature_names_out()
#     print("Transformed feature names:", list(tf_names))
# except Exception:
#     passmodel = CostRegressor(get_input_dim()).to(DEVICE)

Original feature columns: ['CRF05', 'CRF07', 'CRF10', 'CRF11', 'CRF12', 'CRF13', 'CRF14', 'CRF17B', 'CRF17C', 'CRF18', 'CRF19', 'CRF57', 'V106']
Categorical columns: ['CRF05', 'CRF10', 'CRF11', 'CRF12', 'CRF13', 'CRF14', 'CRF17B', 'CRF17C', 'CRF18', 'CRF19', 'CRF57']
Numeric columns: ['CRF07', 'V106']


In [73]:
# ------------------------------ Train the model ----------------------------- #
model = CostRegressor(get_input_dim()).to(DEVICE)
history = train_centralized(model, train_loader, val_loader, 
                            num_epochs=NUM_EPOCHS, 
                            learning_rate=LEARNING_RATE)

model_path = Path.cwd() / 'centralized_model.pt'
torch.save(model.state_dict(), model_path)

print('\nCentralized training complete')
print(f'Saved model to: {model_path}')
print(f'Final train MAE: {history[-1][1]:.4f}')
print(f'Final train MSE: {history[-1][2]:.4f}')
print(f'Final train RMSE: {history[-1][3]:.4f}')
print(f'Final train R2: {history[-1][4]:.4f}')
print(f'Final validation MAE:  {history[-1][5]:.4f}')
print(f'Final validation MSE:  {history[-1][6]:.4f}')
print(f'Final validation RMSE: {history[-1][7]:.4f}')
print(f'Final validation R2:   {history[-1][8]:.4f}')

Epoch 001/150 | train_loss=3124.6692 | train_r2=-0.6236 | val_mae=2446.8934 | val_rmse=4283.2686 | val_r2=-0.4781
Epoch 002/150 | train_loss=2595.2422 | train_r2=-0.4041 | val_mae=1899.8851 | val_rmse=3990.8714 | val_r2=-0.2832
Epoch 003/150 | train_loss=2073.7942 | train_r2=-0.2341 | val_mae=1408.7302 | val_rmse=3763.4796 | val_r2=-0.1411
Epoch 004/150 | train_loss=1650.7540 | train_r2=-0.1298 | val_mae=1150.0845 | val_rmse=3630.5384 | val_r2=-0.0619
Epoch 005/150 | train_loss=1443.8552 | train_r2=-0.0801 | val_mae=1104.0348 | val_rmse=3572.6775 | val_r2=-0.0283
Epoch 006/150 | train_loss=1384.3454 | train_r2=-0.0570 | val_mae=1126.3363 | val_rmse=3548.2644 | val_r2=-0.0143
Epoch 007/150 | train_loss=1364.7788 | train_r2=-0.0447 | val_mae=1150.0549 | val_rmse=3536.1516 | val_r2=-0.0074
Epoch 008/150 | train_loss=1358.7757 | train_r2=-0.0408 | val_mae=1159.8018 | val_rmse=3532.2593 | val_r2=-0.0052
Epoch 009/150 | train_loss=1357.1073 | train_r2=-0.0374 | val_mae=1169.5404 | val_rmse=3

In [74]:
# -------------------------------- Final Test -------------------------------- #
reloaded_model = CostRegressor(get_input_dim()).to(DEVICE)
reloaded_model.load_state_dict(torch.load(model_path, map_location=DEVICE))
test_mae, test_mse, test_rmse, test_r2 = evaluator(reloaded_model, test_loader)

print('Reloaded saved model evaluation on test split')
print(f'MAE:  {test_mae:.4f}')
print(f'MSE:  {test_mse:.4f}')
print(f'RMSE: {test_rmse:.4f}')
print(f'R2:   {test_r2:.4f}')

Reloaded saved model evaluation on test split
MAE:  988.6733
MSE:  8074489.0741
RMSE: 2841.5645
R2:   0.0149
